# Evaluasi CAG, RAG, dan Hybrid CAG-RAG — Wisata Danau Toba

Judul: *Analysis of Pure CAG, Pure RAG, and Hybrid CAG-RAG for Retrieval Latency and Answer Quality*

**Catatan sebelum evaluasi:** bersihkan KV-Cache dulu secara manual dari terminal supaya hasil tidak dipengaruhi cache sebelumnya.

### Dataset Evaluasi (200 query)
| Subset | Jumlah | Sumber |
|--------|--------|--------|
| FAQ (ground-truth) | 100 | `database/FAQ/faq_tourism.json` (random seed=42) |
| Questioner (question-only) | 100 | `database/FAQ/dataset_questioner.json` |

### Inti Skenario
Notebook ini membandingkan 3 framework pada 200 query yang sama:

- **Pure CAG**: cache-based dengan dua layer (**FAQ similarity** lalu **KV-Cache**) tanpa RAG fallback. Implementasi evaluasi tidak mengambil jawaban langsung dari ground-truth (anti-cheating).
- **Pure RAG**: retrieval-only. Semua 200 query dijawab melalui retrieval dari PDF, lalu fallback ke internet bila jawaban belum ditemukan.
- **Hybrid CAG-RAG**: cache-first. Query dicoba via FAQ/KV-Cache terlebih dulu, lalu fallback ke RAG untuk retrieval dari PDF dan internet.

### Metrik Evaluasi

**1. Retrieval Latency Metrics**
- Response Time Avg
- Cache Hit Rate (CHR)
- Speedup Factor

**2. Retrieval Inaccuracy Metrics**
- Relevance Score
- Exact Match (EM)
- Effective Information Rate (EIR)
- RAG Recall
- BERTScore F1
- Completeness
- Hallucination Rate

### Ringkasan Peran Framework
| Framework | Peran Utama | Harapan Hasil |
|-----------|-------------|----------------|
| Pure CAG | FAQ similarity + KV-Cache (tanpa RAG fallback) | Cepat untuk cache hit, miss tetap miss, dan pengujian tetap fair | 
| Pure RAG | Retrieval penuh | Semua query bisa dijawab lewat retrieval PDF + internet fallback |
| Hybrid CAG-RAG | Cache-first + RAG fallback | Menggabungkan kecepatan cache dan cakupan retrieval |


## Setup Environment

In [ ]:
# Install Dependencies
%pip install -q python-dotenv sentence-transformers langchain langchain-community langchain-huggingface faiss-cpu bert-score pandas matplotlib seaborn scipy nltk

In [ ]:
# Setup & Imports
import sys, os, json, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime
from scipy import stats
import scipy
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
from bert_score import score as bert_score_fn
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

# -- Path & env ------------------------------------------------------------
sys.path.append(os.path.abspath('../src'))
load_dotenv()

# -- Import evaluation classes dari evaluation.py -------------------------
from evaluation import QuantitativeMetrics, PerformanceEvaluator, QueryMetrics

# -- Plot style -----------------------------------------------------------
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("✅ Setup complete")
print(f"   scipy        : {scipy.__version__}")
print(f"   evaluation.py: QuantitativeMetrics, PerformanceEvaluator loaded")


## Load Model & Encoder
- LLM: Gemini 2.5 Flash via Google REST API
- Encoder: `paraphrase-multilingual-MiniLM-L12-v2` (multilingual embedding)

In [ ]:
# Load Model & Encoder (digunakan oleh seluruh notebook)
from model import GeminiChatModel
from langchain_huggingface import HuggingFaceEmbeddings

gemini = GeminiChatModel(model_name="gemini-2.5-flash")

encoder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print("Model   : gemini-2.5-flash")
print(f"API keys: {len(gemini.api_keys)}")
print("Encoder : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


In [ ]:
# ============================================================
# QUICK TEST — Gemini 2.5 Flash API
# Jalankan cell ini SEBELUM cell lainnya untuk memastikan
# API key valid dan gemini-2.5-flash dapat merespons.
# Cell ini berdiri sendiri — tidak butuh cell sebelumnya.
# ============================================================
import sys, os, io, contextlib
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "src"))
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv("../.env")

from model import GeminiChatModel

print("=" * 50)
print("Quick Test: Gemini 2.5 Flash API")
print("=" * 50)

gemini_test = GeminiChatModel(model_name="gemini-2.5-flash")

_test_prompt = "Sebutkan satu tempat wisata terkenal di Danau Toba. Jawab dalam 1-2 kalimat."

_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    _resp = gemini_test._call_gemini_api(_test_prompt, max_tokens=100)
_log = _buf.getvalue()

active_model = None
for _line in _log.splitlines():
    if "Response from" in _line:
        active_model = _line.split("Response from")[-1].strip().split()[0]
        break

print(f"\nHasil:")
print(f"  API key     : {len(gemini_test.api_keys)} key ditemukan")
print(f"  Model aktif : {active_model or 'tidak terdeteksi'}")
print(f"  Response    : {repr(_resp[:80]) if _resp else None}")

if _resp and active_model and "2.5-flash" in active_model:
    print("\nSIAP: gemini-2.5-flash berjalan normal, lanjutkan ke cell berikutnya")
elif _resp and active_model:
    print(f"\nFALLBACK: yang merespons {active_model} (bukan 2.5-flash)")
    print("  Kemungkinan: quota 2.5-flash habis atau key belum valid")
else:
    print("\nGAGAL: API tidak merespons")
    print("  Cek: GEMINI_API_KEY di .env sudah benar?")
    print(_log)


## Build Knowledge Base

In [ ]:
# Load Documents (hanya daftar file).
# Indexing + chunking mengikuti backend lewat CAGSystem.load_documents().
tourism_dir = os.path.abspath('../database/documents')

pdf_files = []
if os.path.exists(tourism_dir):
    pdf_files = [
        os.path.join(tourism_dir, f)
        for f in sorted(os.listdir(tourism_dir))
        if f.endswith('.pdf')
    ]

if pdf_files:
    print(f"✅ {len(pdf_files)} PDFs ditemukan di {tourism_dir}")
    for p in pdf_files:
        print(f"   - {os.path.basename(p)}")
else:
    print(f"⚠️  Tidak ada PDF di {tourism_dir}")
    print("   Letakkan file PDF wisata ke folder database/documents/")

# Placeholder vars untuk kompatibilitas cell lama (tidak dipakai).
faiss_db = None
docs = []


## Setup CAG System

In [ ]:
# Setup CAG System (Hybrid CAG-RAG)
try:
    from hybrid_system import CAGSystem   # nama modul yang benar
    cag = CAGSystem(gemini, encoder)
    if pdf_files:
        cag.load_documents(pdf_files, use_summaries=False)
    cag_ok = True
    print("✅ CAG System (Hybrid CAG-RAG) ready")
    print(f"   Confirmed cache entries : {len(cag.kv_cache.cache)}")
    print(f"   Staging entries         : {len(cag.kv_cache.staging)}")
except Exception as e:
    cag_ok = False
    print(f"⚠️  CAG not available: {e}")

## Test Dataset

In [ ]:
# ── Dataset Builder: FAQ + Question-only Questionnaire ──────────────────────
# FAQ tetap dipakai sebagai cache-grounded set.
# Questionnaire sekarang hanya berisi pertanyaan user, tanpa jawaban di file data.
# Atribut tambahan dipakai untuk analisis intent dan routing.

import re as _re_kw
import os as _os
import random as _random
from collections import Counter as _Counter


def _extract_keywords(text: str, top_n: int = 6) -> list:
    stopwords_id = {
        'yang', 'dan', 'dari', 'untuk', 'dengan', 'adalah', 'pada',
        'atau', 'juga', 'akan', 'ini', 'itu', 'bisa', 'dapat', 'lebih',
        'telah', 'sudah', 'sedang', 'dalam', 'oleh', 'antara', 'serta',
        'maka', 'jika', 'agar', 'bagi', 'baik', 'atas', 'tersebut',
        'mereka', 'kami', 'kita', 'saya', 'anda', 'tidak', 'belum',
        'sangat', 'sekali', 'hanya', 'masih', 'sebagai', 'seperti',
        'karena', 'setiap', 'saat', 'yaitu', 'salah', 'satu', 'para',
        'menjadi', 'secara', 'hingga', 'sampai', 'melalui', 'namun',
        'tetapi', 'sehingga', 'dimana', 'ketika', 'selain', 'hampir',
        'nbsp', 'quot',
    }
    from collections import Counter as _C
    tokens = _re_kw.findall(r'[A-Za-z\u00C0-\u024F]{4,}', text)
    freq = _C(w for w in tokens if w.lower() not in stopwords_id)
    seen = {}
    for w in tokens:
        key = w.lower()
        if key not in seen and key not in stopwords_id:
            seen[key] = w
    top_keys = [k for k, _ in freq.most_common(top_n)]
    return [seen[k] for k in top_keys if k in seen]


def _infer_intent(category: str) -> str:
    mapping = {
        'hotel_penginapan': 'lodging',
        'wisata_umum': 'general_tourism',
        'wisata_spesifik': 'specific_attraction',
        'preferensi': 'preference',
        'kuliner': 'culinary',
        'wisata_budaya': 'culture',
        'transportasi': 'transport',
        'penginapan_luar_balige': 'lodging_outside_balige',
        'aktivitas_petualangan': 'adventure',
        'informasi_praktis': 'practical_info',
        'event_festival': 'event_festival',
        'makanan_tradisional': 'traditional_food',
    }
    return mapping.get(category or '', 'general_tourism')


def _infer_difficulty(question: str, category: str) -> str:
    word_count = len((question or '').split())
    if category in {'hotel_penginapan', 'wisata_spesifik', 'transportasi'}:
        base = 'medium'
    elif category in {'preferensi', 'informasi_praktis'}:
        base = 'easy'
    else:
        base = 'medium'
    if word_count > 18:
        return 'hard' if base != 'easy' else 'medium'
    return base


NUM_FAQ_SAMPLES = 100
NUM_QUESTIONER_SAMPLES = 100
FAQ_RANDOM_SEED = 42

# ── Part 1: dataset_faq ─────────────────────────────────────────────────────
_faq_source_path = _os.path.abspath('../database/FAQ/faq_tourism.json')
dataset_faq = []

if _os.path.exists(_faq_source_path):
    with open(_faq_source_path, 'r', encoding='utf-8') as _f:
        _faq_all = json.load(_f)

    _faq_valid = [
        x for x in _faq_all
        if x.get('question', '').strip() and x.get('answer', '').strip()
    ]

    _random.seed(FAQ_RANDOM_SEED)
    _faq_sampled = _random.sample(_faq_valid, min(NUM_FAQ_SAMPLES, len(_faq_valid)))

    dataset_faq = [
        {
            'q':        item['question'],
            'gt':       item['answer'],
            'kw':       _extract_keywords(item['answer']),
            'source':   'faq',
            'category': 'faq',
            'intent':   'faq_static',
        }
        for item in _faq_sampled
    ]
    print(f"✅ dataset_faq         : {len(dataset_faq)} queries")
    print(f"   Sumber              : faq_tourism.json ({len(_faq_valid)}/{len(_faq_all)} entri valid, seed=42)")
    print(f"   Contoh Q            : {dataset_faq[0]['q'][:65]}")
else:
    print(f"⚠️  faq_tourism.json tidak ditemukan di: {_faq_source_path}")


# ── Part 2: dataset_questioner (question-only) ───────────────────────────────
_questioner_path = _os.path.abspath('../database/FAQ/dataset_questioner.json')
dataset_questioner = []

if _os.path.exists(_questioner_path):
    with open(_questioner_path, 'r', encoding='utf-8') as _f:
        _questioner_raw = json.load(_f)

    _questioner_valid = [
        item for item in _questioner_raw
        if item.get('question', '').strip()
    ]
    _random.seed(FAQ_RANDOM_SEED)
    _questioner_sampled = _random.sample(
        _questioner_valid,
        min(NUM_QUESTIONER_SAMPLES, len(_questioner_valid)),
    )

    dataset_questioner = [
        {
            'q':               item.get('question', '').strip(),
            'gt':              None,
            'kw':              [],
            'source':          item.get('source', 'kuesioner'),
            'category':        item.get('category', 'general_tourism'),
            'intent':          item.get('intent') or _infer_intent(item.get('category', 'general_tourism')),
            'difficulty':      item.get('difficulty') or _infer_difficulty(item.get('question', ''), item.get('category', 'general_tourism')),
            'expected_route':  item.get('expected_route', 'HYBRID_RAG_FALLBACK'),
            'evaluation_mode': item.get('evaluation_mode', 'question_only'),
            'cache_policy':    item.get('cache_policy', 'non_faq_miss_first'),
        }
        for item in _questioner_sampled
    ]

    _src_cnt = _Counter(item.get('source', '') for item in dataset_questioner)
    _cat_cnt = _Counter(item.get('category', '') for item in dataset_questioner)
    _intent_cnt = _Counter(item.get('intent', '') for item in dataset_questioner)
    print(f"\n✅ dataset_questioner  : {len(dataset_questioner)} queries")
    print(f"   Mode                : question-only (tanpa jawaban di data user)")
    print(f"   Source              : {dict(_src_cnt)}")
    print(f"   Category            : {dict(_cat_cnt)}")
    print(f"   Intent              : {dict(_intent_cnt)}")
    print(f"   Contoh Q            : {dataset_questioner[0]['q'][:65]}")
else:
    print(f"⚠️  dataset_questioner.json tidak ditemukan di: {_questioner_path}")


# ── Part 3: Gabung & Ringkas ────────────────────────────────────────────────
dataset = dataset_faq + dataset_questioner

print(f"\n{'='*60}")
print(f"📊 DATASET SUMMARY — 100 FAQ + 100 Question-only Questionnaire = 200 Total")
print(f"{'='*60}")
print(f"  dataset_faq          (CAG/FAQ path)   : {len(dataset_faq):>4} queries")
print(f"  dataset_questioner   (question-only)  : {len(dataset_questioner):>4} queries")
print(f"  Total                                 : {len(dataset):>4} queries")
print(f"{'='*60}")
print(f"  Skenario penelitian:")
print(f"  - FAQ            → cache-grounded, answer tersedia")
print(f"  - Questionnaire   → question-only user queries, tanpa answer di data")
print(f"  - Hybrid         → 200 query yang sama diuji di semua framework")
print(f"{'='*60}")
print(f"  Rekomendasi evaluasi:")
print(f"  - Latency dan CHR tetap dihitung untuk semua query")
print(f"  - EM / BERTScore / Completeness pada questionnaire akan N/A bila GT tidak ada")
print(f"  - Hallucination / Relevance / EIR masih bisa dipakai untuk analisis kualitas keluaran")


In [ ]:
# === Benchmark scenario aliases ===
# Evaluasi memakai 200 sampel yang sama untuk semua framework.
# Perbedaannya ada pada router/inference path, bukan pada subset data.
# - PURE_CAG : FAQ similarity + KV-Cache lookup (tanpa RAG fallback)
# - PURE_RAG : retrieval-only routing pada seluruh 200 query
# - HYBRID   : cache-first routing + RAG fallback pada seluruh 200 query

dataset_pure_cag = dataset
dataset_pure_rag = dataset
dataset_hybrid = dataset

benchmark_scenarios = {
    "PURE_CAG": {
        "dataset": dataset_pure_cag,
        "system": "CAG",
        "routing": "faq similarity -> kv-cache lookup; no rag fallback",
        "cache_expectation": "hit rate dihitung terhadap 200 query (FAQ + KV-Cache)",
    },
    "PURE_RAG": {
        "dataset": dataset_pure_rag,
        "system": "RAG",
        "routing": "vector retrieval + generation on 200 queries",
        "cache_expectation": "0% cache hit",
    },
    "HYBRID": {
        "dataset": dataset_hybrid,
        "system": "Hybrid CAG-RAG",
        "routing": "cache-first on 200 queries; RAG fallback on misses; KV warm-up tracked separately",
        "cache_expectation": "CHR utama dihitung pada 200 query pertama",
    },
}

print("✅ Benchmark scenarios ready")
for name, spec in benchmark_scenarios.items():
    print(f"  - {name:<8}: {len(spec['dataset']):>3} queries | {spec['system']} | {spec['routing']}")
print(f"  Total dataset: {len(dataset_hybrid)} queries (same 200 samples for all frameworks)")


## Inference Functions
- `rag_infer()` — pure RAG: retrieve + generate, tanpa cache awal
- `cag_infer()` — hybrid CAG-RAG: cache-first, fallback ke RAG jika miss

In [ ]:
# === Inference Functions ===
#
# RAG  : use_cache=False  → selalu retrieve dari vector DB + generate (tidak pakai cache)
# CAG  : use_cache=True   → cek cache dulu; jika hit → langsung return,
#                           jika miss → retrieve + generate + simpan ke cache
#
# Kedua fungsi menggunakan cag.get_response() yang sama, perbedaannya hanya pada
# flag use_cache. Ini memastikan evaluasi RAG vs CAG apples-to-apples.


def rag_infer(q: str, k: int = 8) -> dict:
    """RAG pure: selalu retrieve + generate, tanpa cache."""
    if not cag_ok:
        return {'resp': '', 'ctx': '', 'time': 0, 'cache': False, 'retrieval_time': 0.0}
    t0 = time.perf_counter()
    r = cag.get_response(q, k=k, use_cache=False)
    return {
        'resp': r.get('response', ''),
        'ctx': r.get('context', ''),
        'time': time.perf_counter() - t0,
        'cache': False,
        'retrieval_time': r.get('retrieval_time', 0.0),
    }


def cag_infer(q: str, k: int = 8) -> dict:
    """CAG: cache-first, lalu RAG jika miss."""
    if not cag_ok:
        return {'resp': '', 'ctx': '', 'time': 0, 'cache': False, 'retrieval_time': 0.0}
    t0 = time.perf_counter()
    r = cag.get_response(q, k=k, use_cache=True)
    return {
        'resp': r.get('response', ''),
        'ctx': r.get('context', ''),
        'time': time.perf_counter() - t0,
        'cache': r.get('cache_used', False),
        'retrieval_time': r.get('retrieval_time', 0.0),
    }


print("✅ Inference ready")
print("   RAG : use_cache=False  (pure retrieval+generate)")
print("   CAG : use_cache=True   (cache-first → RAG fallback)")

---
## ⚙️ Helper: Agregasi Hasil Per-Query

Cell ini menyiapkan fungsi agregasi yang mengumpulkan semua nilai per-query dari `pure_cag_res`, `rag_res`, dan `cag_res` menjadi ringkasan per framework.
Jalankan sekali sebelum semua cell metrik di bawah.

Bagian ini memudahkan perhitungan metrik A dan B sebelum masuk ke visualisasi.

### 1. Response Time Avg

**Definisi:** Rata-rata waktu respons end-to-end sistem per query, dihitung dari saat query diterima hingga jawaban selesai dibangkitkan.

Dalam sistem CAG, total waktu respons mencakup dua tahap:
$$RT = T_{prefill} + (n \times T_{decode})$$

Di mana:
- $T_{prefill}$ = waktu memproses seluruh konteks yang sudah di-cache
- $n$ = jumlah token yang dihasilkan
- $T_{decode}$ = waktu rata-rata untuk menghasilkan satu token

Untuk evaluasi end-to-end antar framework, rata-rata dihitung sebagai:
$$\overline{RT} = \frac{1}{N} \sum_{i=1}^{N} t_i$$

Di mana $t_i$ adalah waktu respons query ke-$i$ dan $N$ adalah total jumlah query.

In [ ]:
# Response Time Avg
# RT_avg = (1/N) * sum(t_i)
def response_time_avg(times: list) -> float:
    if not times:
        return 0.0
    return float(np.mean(times))


def response_time_std(times: list) -> float:
    if not times:
        return 0.0
    return float(np.std(times))


### 2. Cache Hit Rate (CHR)

**Definisi:** Proporsi query yang berhasil dilayani langsung dari cache (FAQ / KV-Cache) tanpa perlu proses retrieval eksternal.

$$\text{CHR} = \frac{\sum_{r=1}^{R} \text{Hits}_r}{\sum_{r=1}^{R} (\text{Hits}_r + \text{Misses}_r)} \times 100\%$$

Di mana:
- $\text{Hits}_r$ = jumlah query yang jawabannya diperoleh dari cache
- $\text{Misses}_r$ = jumlah query yang tidak ditemukan di cache (harus melalui retrieval)
- $R$ = jumlah total pengujian / round evaluasi

CHR tinggi berarti sistem lebih efisien karena sering menghindari retrieval penuh. CHR rendah pada non-FAQ query adalah perilaku yang diharapkan untuk Pure CAG.

In [ ]:
# Cache Hit Rate (CHR)
# CHR = hits / N
def cache_hit_rate(hits: int, total: int) -> float:
    if total <= 0:
        return 0.0
    return hits / total


### 3. Speedup Factor

**Definisi:** Rasio kecepatan antara kondisi cache miss (baseline RAG) terhadap cache hit pada sistem CAG/Hybrid, untuk mengukur seberapa besar percepatan yang dicapai.

Definisi umum speedup:
$$n = \frac{\text{Execution Time}_{old}}{\text{Execution Time}_{new}}$$

Dalam konteks penelitian ini:
$$\text{Speedup Factor} = \frac{T_{miss}}{T_{hit}}$$

Di mana:
- $T_{miss}$ = rata-rata waktu respons query yang mengalami cache miss (proses RAG penuh)
- $T_{hit}$ = rata-rata waktu respons query yang mengalami cache hit (CAG via cache)

Nilai $> 1.0$ menunjukkan bahwa sistem cache lebih cepat dari baseline RAG.

In [ ]:
# Speedup Factor
# Speedup = RT_avg_RAG / RT_avg_target
def speedup_factor(baseline_avg: float, target_avg: float) -> float:
    if target_avg is None or target_avg <= 0:
        return float('nan')
    if baseline_avg is None or baseline_avg <= 0:
        return float('nan')
    return baseline_avg / target_avg


### 4. Relevance Score

**Definisi:** Mengukur sejauh mana jawaban yang dihasilkan secara langsung merespons dan sesuai dengan pertanyaan yang diberikan, tanpa mempertimbangkan faktualitas konten.

$$AR = \frac{1}{n} \sum_{i=1}^{n} \text{sim}(q, q_i)$$

Di mana:
- $n$ = jumlah pertanyaan potensial yang dihasilkan dari jawaban
- $q$ = pertanyaan asli pengguna
- $q_i$ = pertanyaan ke-$i$ yang diekstrak dari jawaban model
- $\text{sim}(q, q_i)$ = cosine similarity antara embedding $q$ dan $q_i$

Dalam implementasi ini, cosine similarity dihitung langsung antara embedding jawaban dan embedding query sebagai aproksimasi efisien:
$$\text{Rel}(r, q) = \cos(\vec{r}, \vec{q}) = \frac{\vec{r} \cdot \vec{q}}{\|\vec{r}\| \|\vec{q}\|}$$

In [ ]:
import re as _re
from collections import Counter
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
qm = QuantitativeMetrics()

# Relevance Score
# Rel(r, q) = cos(r_emb, q_emb)
def relevance_score(resp: str, query: str) -> float:
    result = qm.calculate_irrelevancy_score(resp, query)
    return result["relevance_score"]


### 5. Exact Match (EM)

**Definisi:** Mengukur tingkat kesesuaian string antara jawaban yang dihasilkan model dengan jawaban referensi (ground truth) setelah normalisasi teks.

$$\text{EM}(pred, gold) = \begin{cases} 1 & \text{jika } \text{normalize}(pred) = \text{normalize}(gold) \\ 0 & \text{jika tidak} \end{cases}$$

**Langkah normalisasi:**
1. Konversi ke huruf kecil (*lowercase*)
2. Hapus *stopwords* (bahasa Indonesia dan Inggris)
3. Hapus karakter non-alfanumerik
4. Normalisasi spasi

**Catatan:** EM hanya dihitung pada subset FAQ (100 query dengan ground truth). Subset questioner tidak memiliki ground truth sehingga nilainya NaN.

Rata-rata EM dihitung dari query yang memiliki ground truth:
$$\overline{EM} = \frac{1}{N_{gt}} \sum_{i=1}^{N_{gt}} \text{EM}(pred_i, gold_i)$$

In [ ]:
# Exact Match (EM)
def _normalize_text(s: str) -> str:
    s = s.lower()
    s = _re.sub(r'\b(a|an|the|dan|yang|di|ke|dari|untuk|adalah)\b', ' ', s)
    s = _re.sub(r'[^a-z0-9\s]', ' ', s)
    return ' '.join(s.split())


def exact_match(pred: str, gold: str) -> float:
    return 1.0 if _normalize_text(pred) == _normalize_text(gold) else 0.0


def accuracy_score(resp: str, gt: str) -> dict:
    return {'em': exact_match(resp, gt)}


### 6. Effective Information Rate (EIR)

**Definisi:** Mengukur proporsi informasi yang relevan dari keseluruhan teks yang berhasil diambil oleh sistem retrieval. Metrik ini memastikan retrieval tidak hanya akurat dalam menemukan informasi benar, tetapi juga efisien dengan meminimalkan konten tidak relevan (*noise*).

$$\text{EIR} = \frac{\sum_{i=1}^{m} |G_i \cap R_t|}{\sum_{j=1}^{k} |R_j|}$$

Di mana:
- $G_i$ = ground truth reference ke-$i$ (kalimat/kata kunci dalam jawaban benar)
- $R_t$ = himpunan semua token/kata dalam konteks yang diambil sistem
- $R_j$ = passage ke-$j$ yang berhasil diambil sistem
- $m$ = jumlah referensi ground truth
- $k$ = jumlah passage yang diambil

Dalam implementasi token-level:
$$\text{EIR}(c, r) = \frac{|C_w \cap R_w|}{|C_w|}$$

**Catatan:** EIR bernilai NaN untuk query FAQ (konteks dari cache, bukan retrieval) dan untuk query tanpa ground truth pada subset questioner.

In [ ]:
# Effective Information Rate (EIR)
# EIR(c, r) = |C_w ∩ R_w| / |C_w|
def eir(ctx: str, resp: str) -> float:
    _ctx = (ctx or "").strip()
    if not _ctx or _ctx == "from_faq" or len(_ctx) < 20:
        return float('nan')
    if not resp:
        return 0.0
    cw = {w.lower() for w in _ctx.split() if len(w) > 4}
    rw = {w.lower() for w in resp.split() if len(w) > 4}
    return len(cw & rw) / len(cw) if cw else 0.0


### 7. RAG Recall

**Definisi:** Proporsi keyword ground-truth yang berhasil ditemukan dalam konteks hasil retrieval. Metrik ini mengukur kelengkapan pengambilan informasi dari dokumen.

$$\text{Recall}_{RAG} = \frac{1}{n} \sum_{i=1}^{n} M(G_i, R)$$

Di mana:
- $n$ = jumlah total referensi ground truth
- $G_i$ = referensi ground truth ke-$i$
- $R = \{R_1, R_2, \ldots, R_k\}$ = himpunan dokumen/konteks yang diperoleh
- $M(G_i, R)$ = fungsi boolean: 1 jika $G_i$ ditemukan dalam minimal satu referensi di $R$

Dalam implementasi keyword-level:
$$\text{Recall}_{RAG}(c, KW) = \frac{|\{k \in KW : k \in c\}|}{|KW|}$$

Di mana $c$ adalah konteks retrieval dan $KW$ adalah himpunan keyword ground-truth.

**Catatan:** RAG Recall hanya bermakna untuk subset questioner yang melalui jalur retrieval (non-FAQ). Untuk Pure CAG, nilai ini NaN karena tidak ada retrieval.

In [ ]:
# RAG Recall
# Recall_RAG(c, KW) = |{k in KW : k in c}| / |KW|
def rag_recall(ctx: str, kw: list) -> float:
    _ctx = (ctx or "").strip()
    if not _ctx or _ctx == "from_faq" or len(_ctx) < 20:
        return float('nan')
    if not kw:
        return 0.0
    ctx_lower = _ctx.lower()
    return sum(k.lower() in ctx_lower for k in kw) / len(kw)


### 8. BERTScore F1

**Definisi:** Mengukur kemiripan semantik antara jawaban yang dihasilkan model dan ground truth menggunakan representasi vektor token dari model BERT.

**BERTScore Precision** — seberapa relevan token kandidat terhadap referensi:
$$P_{BERT} = \frac{1}{|\hat{x}|} \sum_{\hat{x}_j \in \hat{x}} \max_{x_i \in x} v_{x_i}^\top v_{\hat{x}_j}$$

**BERTScore Recall** — seberapa lengkap token referensi terwakili di kandidat:
$$R_{BERT} = \frac{1}{|x|} \sum_{x_i \in x} \max_{\hat{x}_j \in \hat{x}} v_{x_i}^\top v_{\hat{x}_j}$$

**BERTScore F1** — harmonik mean dari Precision dan Recall:
$$F1_{BERT} = 2 \times \frac{P_{BERT} \cdot R_{BERT}}{P_{BERT} + R_{BERT}}$$

Di mana:
- $x$ = urutan token referensi (ground truth)
- $\hat{x}$ = urutan token kandidat (jawaban model)
- $v_{x_i}, v_{\hat{x}_j}$ = vektor embedding token dari model BERT
- $v_{x_i}^\top v_{\hat{x}_j}$ = cosine similarity antar token

**Catatan:** Hanya dihitung pada query dengan ground truth (subset FAQ, 100 query).

In [ ]:
# BERTScore F1
# F_BERT = 2 * P_BERT * R_BERT / (P_BERT + R_BERT)
def bertscore_f1(preds: list, refs: list) -> float:
    if not preds or not refs or not preds[0] or not refs[0]:
        return 0.0
    try:
        P, R, F = bert_score_fn(preds, refs, lang='id', verbose=False)
        return float(F.mean().item())
    except Exception as e:
        print(f"  ⚠️  BERTScore error: {e}")
        return 0.0


### 9. Completeness

**Definisi:** Mengukur seberapa baik jawaban yang dihasilkan menangkap key points dari ground truth. Metrik ini mengevaluasi kelengkapan informasi dalam jawaban.

$$\text{Comp}(A, K) = \frac{1}{|K|} \sum_{i=1}^{|K|} \mathbb{1}[A \text{ covers } k_i]$$

Di mana:
- $A$ = jawaban yang dihasilkan model
- $K = \{k_1, k_2, \ldots, k_n\}$ = himpunan key points dari ground truth
- $\mathbb{1}[\cdot]$ = fungsi indikator: 1 jika jawaban secara semantik mencakup $k_i$, 0 jika tidak
- *Covers* = jawaban mengandung informasi yang konsisten dan benar tentang $k_i$

Dalam implementasi keyword-level:
$$\text{Comp}(r, KW) = \frac{|\{k \in KW : k \in r\}|}{|KW|}$$

Di mana $KW$ adalah keyword yang diekstrak dari ground truth dan $r$ adalah respons model.

**Catatan:** Hanya dihitung pada query dengan ground truth (subset FAQ).

In [ ]:
# Completeness
# Comp(r, KW) = |{k in KW : k in r}| / |KW|
def completeness(resp: str, kw: list) -> float:
    result = qm.calculate_completeness(resp, kw)
    return result["completeness"]


### 10. Hallucination Rate

**Definisi:** Mengukur proporsi informasi dalam jawaban yang bertentangan dengan fakta-fakta kunci dari sumber aslinya. Dalam konteks RAG, ini mengukur seberapa tidak *grounded* jawaban terhadap konteks yang diambil.

Definisi berbasis key points:
$$\text{Hallu}(A, K) = \frac{1}{|K|} \sum_{i=1}^{|K|} \mathbb{1}[A \text{ contradicts } k_i]$$

Hubungan dengan Completeness dan Irrelevancy:
$$\text{Irr}(A, K) = 1 - \text{Comp}(A, K) - \text{Hallu}(A, K)$$

Dalam implementasi berbasis embedding cosine distance:
$$\text{Hall}(r, c) = 1 - \cos(\vec{r}, \vec{c}) = 1 - \frac{\vec{r} \cdot \vec{c}}{\|\vec{r}\| \|\vec{c}\|}$$

Di mana $r$ = embedding jawaban dan $c$ = embedding konteks retrieval.

Nilai mendekati 0 berarti jawaban sangat *grounded*; nilai mendekati 1 berarti potensi halusinasi tinggi.

**Catatan:** Bernilai NaN untuk query FAQ (konteks dari cache, bukan retrieval).

In [ ]:
# Hallucination Rate
# Hall(r, c) = 1 - cos(r_emb, c_emb)
def hallucination(resp: str, ctx: str) -> float:
    _ctx = (ctx or "").strip()
    if not _ctx or _ctx == "from_faq" or len(_ctx) < 20:
        return float('nan')

    try:
        resp_emb = np.array(encoder.embed_query(resp[:512]))
        ctx_emb = np.array(encoder.embed_query(_ctx[:512]))
        norm = np.linalg.norm(resp_emb) * np.linalg.norm(ctx_emb)
        cos_sim = float(np.dot(resp_emb, ctx_emb) / (norm + 1e-9))
        return float(np.clip(1.0 - cos_sim, 0.0, 1.0))
    except Exception:
        pass

    def _bigrams(text: str) -> set:
        tokens = _re.sub(r'[^\w\s]', ' ', text.lower()).split()
        return set(zip(tokens[:-1], tokens[1:])) if len(tokens) > 1 else set()

    resp_bg = _bigrams(resp)
    ctx_bg = _bigrams(_ctx)
    if not resp_bg:
        return 0.0
    overlap = len(resp_bg & ctx_bg) / len(resp_bg)
    return float(1.0 - overlap)


### 11. Statistical Significance Tests

Dipakai setelah hasil metrik utama dihitung.
- Paired t-test membandingkan dua daftar skor berpasangan.
- Cohen's d mengukur besar efek perbedaannya.

Bagian ini dipakai untuk menguji apakah perbedaan antara RAG dan Hybrid CAG-RAG signifikan secara statistik.

In [ ]:
# Statistical helpers
# paired_ttest: Student's paired t-test
# cohens_d: standardized effect size
def paired_ttest(a: list, b: list):
    pairs = [
        (x, y) for x, y in zip(a, b)
        if not (isinstance(x, float) and np.isnan(x))
        and not (isinstance(y, float) and np.isnan(y))
    ]
    if len(pairs) < 2:
        return (float('nan'), float('nan'), len(pairs))
    va, vb = zip(*pairs)
    t, p = stats.ttest_rel(list(va), list(vb))
    return (round(float(t), 4), round(float(p), 4), len(pairs))


def cohens_d(a: list, b: list) -> float:
    pairs = [
        (x, y) for x, y in zip(a, b)
        if not (isinstance(x, float) and np.isnan(x))
        and not (isinstance(y, float) and np.isnan(y))
    ]
    if len(pairs) < 2:
        return float('nan')
    va, vb = zip(*pairs)
    diff = [x - y for x, y in zip(va, vb)]
    return round(float(np.mean(diff) / (np.std(diff, ddof=1) + 1e-9)), 4)


## Evaluation Loop: Three-Framework Benchmark (CAG | RAG | Hybrid CAG-RAG)

Benchmark ini membandingkan tiga framework pada 200 query (100 FAQ + 100 question-only).

| Framework | Jalur Utama | Perilaku |
|-----------|-------------|----------|
| **CAG (Pure)** | FAQ similarity -> KV-Cache (tanpa RAG fallback) | Query dicocokkan ke FAQ secara semantic-hybrid terlebih dulu; jika miss, cek KV-Cache; jika tetap miss maka tidak melakukan retrieval RAG (anti-cheating). |
| **RAG (Pure)** | Retrieval + generation | Semua 200 query dijawab via retrieval dari PDF, lalu fallback internet bila perlu. |
| **Hybrid CAG-RAG** | Cache-first -> RAG fallback | FAQ/KV-Cache hit cepat; miss akan fallback ke RAG untuk retrieval dari PDF dan internet. |

All per-query metrics (`bert`, `comp`, `rel`, `hall`, `recall`, `eir`) are computed immediately after each inference call and stored in `r['metrics']` to avoid redundant recomputation in downstream aggregation cells.

In [ ]:
# =============================================================================
# Evaluation Loop — 200-Query Full Benchmark (CAG vs RAG vs Hybrid)
#
# All 200 queries are evaluated on all frameworks.
# FAQ items have ground truth answers.
# Questionnaire items are question-only user queries, so answer-based metrics
# become NaN for those rows unless a separate annotation file is supplied.
# =============================================================================

rag_res = []
pure_cag_res = []
cag_res = []  # Hybrid CAG-RAG results
framework_results = {'CAG': [], 'RAG': [], 'HYBRID': []}


def _compute_metrics(resp, ctx, gt, kw, query):
    """Compute per-query metrics with NaN for unavailable ground truth metrics."""
    gt_available = bool(gt and str(gt).strip())
    kw_available = bool(kw)
    response_available = bool(resp and str(resp).strip())

    return {
        'bert': bertscore_f1([resp], [gt]) if gt_available and response_available else float('nan'),
        'comp': completeness(resp, kw) if gt_available and kw_available and response_available else float('nan'),
        'rel': relevance_score(resp, query) if response_available else float('nan'),
        'hall': hallucination(resp, ctx),
        'recall': rag_recall(ctx, kw) if gt_available and kw_available else float('nan'),
        'eir': eir(ctx, resp) if response_available and ctx else float('nan'),
        'em': exact_match(resp, gt) if gt_available and response_available else float('nan'),
    }


def pure_cag_infer(q: str) -> dict:
    """Pure CAG: FAQ similarity -> KV-Cache lookup only (no RAG fallback)."""
    if not cag_ok:
        return {
            'resp': '',
            'ctx': '',
            'time': 0.0,
            'cache': False,
            'retrieval_time': 0.0,
            'framework': 'CAG',
            'cache_source': 'none',
        }

    t0 = time.perf_counter()

    # 1) FAQ layer: gunakan matcher internal (exact + semantic hybrid), bukan GT lookup.
    faq_hit = cag._search_faq(q)
    if faq_hit and faq_hit.get('answer'):
        ans = faq_hit.get('answer', '')
        try:
            cag.kv_cache.put(q, ans, 'from_faq')
        except Exception:
            pass
        return {
            'resp': ans,
            'ctx': 'from_faq',
            'time': time.perf_counter() - t0,
            'cache': True,
            'retrieval_time': 0.0,
            'framework': 'CAG',
            'cache_source': 'faq_cache',
        }

    # 2) Dynamic cache layer: KV-Cache lookup tanpa retrieval RAG.
    cached = None
    try:
        cached = cag.kv_cache.get(q, context_key='')
    except Exception:
        cached = None

    if cached and cached.get('response'):
        return {
            'resp': cached.get('response', ''),
            'ctx': cached.get('context', ''),
            'time': time.perf_counter() - t0,
            'cache': True,
            'retrieval_time': 0.0,
            'framework': 'CAG',
            'cache_source': 'kv_cache',
        }

    # 3) Hard miss: Pure CAG tidak boleh fallback ke RAG (anti-cheating).
    return {
        'resp': '',
        'ctx': '',
        'time': time.perf_counter() - t0,
        'cache': False,
        'retrieval_time': 0.0,
        'framework': 'CAG',
        'cache_source': 'miss',
    }


def _fmt_secs(sec: float) -> str:
    # Tampilkan presisi mikrodetik agar cache-hit super cepat tidak terlihat 0.00s.
    return f"{sec:.6f}s ({sec * 1000:.3f} ms)"


print(f"Evaluating {len(dataset)} queries  |  CAG vs RAG vs Hybrid CAG-RAG")
print("-" * 72)

for i, d in enumerate(dataset, 1):
    q = d['q']
    gt = d.get('gt', None)
    kw = d.get('kw', [])
    print(f"[{i:>2}/{len(dataset)}] {q[:70]}")

    # ── (A) Pure CAG: FAQ similarity + KV-Cache lookup (tanpa RAG fallback) ──
    pc = pure_cag_infer(q)
    pc.update({'gt': gt, 'kw': kw, 'query': q})
    pc['metrics'] = _compute_metrics(pc.get('resp', ''), pc.get('ctx', ''), gt, kw, q)
    pure_cag_res.append(pc)
    framework_results['CAG'].append(pc)
    print(f"  CAG      : {_fmt_secs(pc['time'])}  cache={'HIT' if pc.get('cache') else 'MISS'}  src={pc.get('cache_source', 'n/a')}")

    # ── (B) Pure RAG: retrieve + generate, no cache ─────────────────────────
    r = rag_infer(q)
    r.update({'gt': gt, 'kw': kw, 'query': q, 'framework': 'RAG'})
    r['metrics'] = _compute_metrics(r.get('resp', ''), r.get('ctx', ''), gt, kw, q)
    rag_res.append(r)
    framework_results['RAG'].append(r)
    print(f"  RAG      : {_fmt_secs(r['time'])}  ctx={len(r.get('ctx', ''))} chars")

    time.sleep(2)

    if cag_ok:
        # ── (C) Hybrid CAG-RAG: cache-first, RAG fallback on miss ─────────────
        c = cag_infer(q)
        c.update({'gt': gt, 'kw': kw, 'query': q, 'framework': 'HYBRID'})
        c['metrics'] = _compute_metrics(c.get('resp', ''), c.get('ctx', ''), gt, kw, q)
        cag_res.append(c)
        framework_results['HYBRID'].append(c)
        if c.get('cache'):
            print(f"  HYBRID   : {_fmt_secs(c['time'])}  cache=HIT")
        else:
            print(f"  HYBRID   : {_fmt_secs(c['time'])}  cache=MISS -> RAG fallback  ctx={len(c.get('ctx', ''))} chars")

    if i < len(dataset):
        time.sleep(1)

print("-" * 72)
print(f"Done.  CAG={len(pure_cag_res)}  RAG={len(rag_res)}  HYBRID={len(cag_res)}")
if pure_cag_res:
    cag_hits = sum(1 for x in pure_cag_res if x.get('cache'))
    print(f"  CAG cache hits   : {cag_hits}/{len(pure_cag_res)}")
if cag_res:
    hybrid_hits = sum(1 for x in cag_res if x.get('cache'))
    print(f"  Hybrid cache hits: {hybrid_hits}/{len(cag_res)}")


In [ ]:
# =============================================================================
# Primary Aggregation (CAG vs RAG vs Hybrid)
#   pure_cag_m — Pure CAG (FAQ-only) on the full 200-query set
#   rag_m      — Pure RAG baseline on the full 200-query set
#   cag_m      — Hybrid CAG-RAG on the full 200-query set
# =============================================================================
pure_cag_m = calc_metrics(pure_cag_res)                 if pure_cag_res else None
rag_m      = calc_metrics(rag_res)
cag_m      = calc_metrics(cag_res)                      if cag_res else None

speedup_hybrid = (rag_m['time_avg'] / cag_m['time_avg']) if (cag_m and cag_m['time_avg'] > 0) else None
speedup_cag    = (rag_m['time_avg'] / pure_cag_m['time_avg']) if (pure_cag_m and pure_cag_m['time_avg'] > 0) else None


def _fmt(val, pct=False, digits=4):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "N/A"
    return f"{val*100:.1f}%" if pct else f"{val:.{digits}f}"


def _fmt_speed(val):
    if val is None or (isinstance(val, float) and np.isnan(val)) or val <= 0:
        return "N/A"
    return f"{val:.2f}x"


print("✅ Metrics calculated\n")
print("=" * 78)
print("📊 TABLE 1 — CAG vs RAG vs Hybrid CAG-RAG (200-query benchmark)")
print("=" * 78)
header_line = f"  {'Metric':<28} {'RAG':>12}  {'CAG (FAQ)':>12}  {'Hybrid':>12}"
print(header_line); print("  " + "-" * 74)

table_rows = [
    ("Response Time Avg (s)", "time_avg", False),
    ("Cache Hit Rate (CHR)", "chr", True),
    ("Speedup Factor", None, False),
    ("Relevance Score", "rel", False),
    ("Exact Match (EM)", "em", False),
    ("EIR", "eir", False),
    ("RAG Recall", "recall", False),
    ("BERTScore F1", "bert", False),
    ("Completeness", "comp", False),
    ("Hallucination Rate", "hall", False),
]

for label, key, is_pct in table_rows:
    if key is None:
        rv = "1.00x"
        cv = _fmt_speed(speedup_cag)
        hv = _fmt_speed(speedup_hybrid)
    else:
        rv = _fmt(rag_m.get(key), pct=is_pct) if rag_m else "N/A"
        cv = _fmt(pure_cag_m.get(key), pct=is_pct) if pure_cag_m else "N/A"
        hv = _fmt(cag_m.get(key), pct=is_pct) if cag_m else "N/A"
    print(f"  {label:<28} {rv:>12}  {cv:>12}  {hv:>12}")

if pure_cag_res:
    cag_hits = sum(1 for x in pure_cag_res if x.get('cache'))
    print(f"\n  CAG cache hits   : {cag_hits}/{len(pure_cag_res)}")
if cag_res:
    hybrid_hits = sum(1 for x in cag_res if x.get('cache'))
    print(f"  Hybrid cache hits: {hybrid_hits}/{len(cag_res)}")
print("=" * 78)


In [ ]:
# Per-Query Detail Tables (CAG vs RAG vs Hybrid)
if pure_cag_m and pure_cag_m.get('per_query'):
    print("\n📄 Per-Query Detail (CAG - FAQ similarity + KV-Cache):")
    df_pq_cag = pd.DataFrame(pure_cag_m['per_query'])
    print(df_pq_cag.to_string(index=False))

if rag_m.get('per_query'):
    print("\n📄 Per-Query Detail (RAG):")
    df_pq_rag = pd.DataFrame(rag_m['per_query'])
    print(df_pq_rag.to_string(index=False))

if cag_m and cag_m.get('per_query'):
    print("\n📄 Per-Query Detail (Hybrid CAG-RAG):")
    df_pq_hybrid = pd.DataFrame(cag_m['per_query'])
    print(df_pq_hybrid.to_string(index=False))


## Table 2 — Framework Comparison (CAG vs RAG vs Hybrid CAG-RAG)

Tabel ini membandingkan tiga framework utama pada 200 query.

- **CAG (Pure)**: FAQ similarity -> KV-Cache (tanpa RAG fallback), sehingga tidak ada jalur retrieval tambahan yang berpotensi cheating.
- **RAG (Pure)**: retrieval dari PDF, lalu fallback internet bila perlu; semua 200 query dijawab.
- **Hybrid CAG-RAG**: cache-first (FAQ/KV-Cache), lalu RAG fallback untuk miss; kombinasi paling kuat untuk latency dan kualitas.

Tabel ringkas per framework ada pada sel berikutnya.

In [ ]:
# =============================================================================
# Framework Comparison: CAG vs RAG vs Hybrid CAG-RAG
# =============================================================================

framework_metrics = {}
if pure_cag_m:
    framework_metrics['CAG'] = pure_cag_m
framework_metrics['RAG'] = rag_m
if cag_m:
    framework_metrics['HYBRID'] = cag_m

framework_order = [f for f in ['CAG', 'RAG', 'HYBRID'] if f in framework_metrics]


def _fmt(val, pct=False):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "N/A"
    return f"{val*100:.1f}%" if pct else f"{val:.4f}"


metric_labels = [
    ("Response Time Avg (s)", 'time_avg', False),
    ("Cache Hit Rate (CHR)",  'chr', True),
    ("Relevance Score",       'rel', False),
    ("Exact Match (EM)",      'em', False),
    ("EIR",                   'eir', False),
    ("RAG Recall",            'recall', False),
    ("BERTScore F1",          'bert', False),
    ("Completeness",          'comp', False),
    ("Hallucination Rate",    'hall', False),
]

print("\n" + "=" * 85)
print("📊 FRAMEWORK COMPARISON: CAG vs RAG vs Hybrid CAG-RAG")
print("=" * 85)

header = "  {:<26} ".format("Metric") + "  ".join([f"{fw:>12}" for fw in framework_order])
print(header)
print("  " + "-" * 82)

for lbl, key, is_pct in metric_labels:
    row_vals = []
    for fw in framework_order:
        v = framework_metrics[fw].get(key)
        row_vals.append(_fmt(v, pct=is_pct))
    print("  {:<26} ".format(lbl) + "  ".join([f"{v:>12}" for v in row_vals]))

# Query count per framework
counts = {
    'CAG': len(pure_cag_res) if pure_cag_res else 0,
    'RAG': len(rag_res) if rag_res else 0,
    'HYBRID': len(cag_res) if cag_res else 0,
}
print("\n  {:<26} ".format("N queries") + "  ".join([f"{counts[fw]:>12}" for fw in framework_order]))
print("=" * 85)
print("\n  Notes:")
print("     CAG: FAQ similarity + KV-Cache only (no RAG fallback in pure setting)")
print("     Hybrid: cache-first with RAG fallback on miss")


In [ ]:
# Per-Framework Visualization: CAG vs RAG vs Hybrid
# Setiap panel juga disimpan sebagai gambar mandiri (individual) untuk keperluan paper.

frameworks_available = [f for f in ['CAG', 'RAG', 'HYBRID'] if f in framework_metrics]
colors = {'CAG': '#2ecc71', 'RAG': '#3498db', 'HYBRID': '#e74c3c'}


def _safe(v):
    return 0.0 if v is None or (isinstance(v, float) and np.isnan(v)) else v


def _label(v):
    return "N/A" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"


width = 0.75 / max(len(frameworks_available), 1)

# ── 1. Latency per Framework (individual figure) ─────────────────────────
fig1, ax1 = plt.subplots(figsize=(6, 5))
times_raw = [framework_metrics[s]['time_avg'] for s in frameworks_available]
times = [_safe(v) for v in times_raw]
bars = ax1.bar(frameworks_available, times,
               color=[colors[s] for s in frameworks_available], alpha=0.85, edgecolor='white')
ax1.set_ylabel('Response Time (s)', fontsize=12)
ax1.set_title('Latency per Framework', fontsize=13, fontweight='bold')
ax1.tick_params(labelsize=11)
for bar, val in zip(bars, times_raw):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             _label(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
fig1.tight_layout()
fig1.savefig('../logs/framework_latency.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/framework_latency.png")

# ── 2. Quality Metrics per Framework (individual figure) ─────────────────
fig2, ax2 = plt.subplots(figsize=(8, 5))
quality_metrics = ['bert', 'comp', 'rel']
quality_labels  = ['BERTScore F1', 'Completeness', 'Relevance']
x_q = np.arange(len(quality_metrics))
for idx, fw in enumerate(frameworks_available):
    raw_vals = [framework_metrics[fw][m] for m in quality_metrics]
    vals = [_safe(v) for v in raw_vals]
    pos = x_q + (idx - (len(frameworks_available) - 1) / 2) * width
    bars2 = ax2.bar(pos, vals, width, label=fw, color=colors[fw], alpha=0.85,
                    edgecolor='white')
    for bar, val in zip(bars2, raw_vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 _label(val), ha='center', va='bottom', fontsize=8)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Quality Metrics per Framework', fontsize=13, fontweight='bold')
ax2.set_xticks(x_q)
ax2.set_xticklabels(quality_labels, rotation=15, fontsize=11)
ax2.tick_params(labelsize=11)
ax2.legend(fontsize=11)
ax2.set_ylim(0, 1.12)
ax2.grid(axis='y', alpha=0.3)
fig2.tight_layout()
fig2.savefig('../logs/framework_quality.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/framework_quality.png")

# ── 3. Inaccuracy Metrics per Framework (individual figure) ──────────────
fig3, ax3 = plt.subplots(figsize=(7, 5))
inacc_metrics = ['hall']
inacc_labels  = ['Hallucination Rate']
x_i = np.arange(len(inacc_metrics))
for idx, fw in enumerate(frameworks_available):
    raw_vals = [framework_metrics[fw][m] for m in inacc_metrics]
    vals = [_safe(v) for v in raw_vals]
    pos = x_i + (idx - (len(frameworks_available) - 1) / 2) * width
    bars3 = ax3.bar(pos, vals, width, label=fw, color=colors[fw], alpha=0.85,
                    edgecolor='white')
    for bar, val in zip(bars3, raw_vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 _label(val), ha='center', va='bottom', fontsize=9)
ax3.set_ylabel('Rate (lower = better)', fontsize=12)
ax3.set_title('Inaccuracy Metrics per Framework', fontsize=13, fontweight='bold')
ax3.set_xticks(x_i)
ax3.set_xticklabels(inacc_labels, fontsize=11)
ax3.tick_params(labelsize=11)
ax3.legend(fontsize=11)
ax3.grid(axis='y', alpha=0.3)
fig3.tight_layout()
fig3.savefig('../logs/framework_inaccuracy.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/framework_inaccuracy.png")

# ── Combined overview (3-panel) ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# panel 1: latency
ax = axes[0]
bars = ax.bar(frameworks_available, times, color=[colors[s] for s in frameworks_available], alpha=0.85)
ax.set_ylabel('Response Time (s)', fontsize=11)
ax.set_title('Latency per Framework', fontsize=12, fontweight='bold')
ax.tick_params(labelsize=10)
for bar, val in zip(bars, times_raw):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            _label(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# panel 2: quality
ax = axes[1]
for idx, fw in enumerate(frameworks_available):
    raw_vals = [framework_metrics[fw][m] for m in quality_metrics]
    vals = [_safe(v) for v in raw_vals]
    pos = x_q + (idx - (len(frameworks_available) - 1) / 2) * width
    ax.bar(pos, vals, width, label=fw, color=colors[fw], alpha=0.85)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Quality Metrics per Framework', fontsize=12, fontweight='bold')
ax.set_xticks(x_q)
ax.set_xticklabels(quality_labels, rotation=15, fontsize=10)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.12)
ax.grid(axis='y', alpha=0.3)

# panel 3: inaccuracy
ax = axes[2]
for idx, fw in enumerate(frameworks_available):
    raw_vals = [framework_metrics[fw][m] for m in inacc_metrics]
    vals = [_safe(v) for v in raw_vals]
    pos = x_i + (idx - (len(frameworks_available) - 1) / 2) * width
    ax.bar(pos, vals, width, label=fw, color=colors[fw], alpha=0.85)
ax.set_ylabel('Rate (lower = better)', fontsize=11)
ax.set_title('Inaccuracy Metrics per Framework', fontsize=12, fontweight='bold')
ax.set_xticks(x_i)
ax.set_xticklabels(inacc_labels, fontsize=10)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../logs/framework_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/framework_comparison.png (overview)")


## Latency Analysis (CAG vs RAG vs Hybrid)

Sesi ini membahas perbandingan latensi antar framework untuk 200 query. Tujuannya adalah melihat bukan hanya rata-rata, tetapi juga stabilitas respons per query.

Urutan visual di bawah ini:

1. **Response Time per Query** — timeline waktu respons setiap query untuk CAG, RAG, dan Hybrid
2. **Avg Response Time per Framework** — rata-rata dan deviasi standar tiap framework
3. **Speedup Factor** — perbandingan rata-rata RAG vs CAG, dan RAG vs Hybrid

In [ ]:
# LATENCY ANALYSIS — Framework Comparison (CAG vs RAG vs Hybrid)

rag_times     = rag_m['_time']
cag_times     = pure_cag_m['_time'] if pure_cag_m else []
hybrid_times  = cag_m['_time'] if cag_m else []

speedup_cag    = (rag_m['time_avg'] / pure_cag_m['time_avg']) if (pure_cag_m and pure_cag_m['time_avg'] > 0) else None
speedup_hybrid = (rag_m['time_avg'] / cag_m['time_avg']) if (cag_m and cag_m['time_avg'] > 0) else None

print("=" * 62)
print("LATENCY SUMMARY — CAG vs RAG vs HYBRID")
print("=" * 62)
print(f"  RAG avg time     : {rag_m['time_avg']:.3f}s")
if pure_cag_m:
    print(f"  CAG avg time     : {pure_cag_m['time_avg']:.3f}s")
    print(f"  Speedup (RAG/CAG): {speedup_cag:.2f}x" if speedup_cag else "  Speedup (RAG/CAG): N/A")
if cag_m:
    print(f"  Hybrid avg time  : {cag_m['time_avg']:.3f}s")
    print(f"  Speedup (RAG/Hybrid): {speedup_hybrid:.2f}x" if speedup_hybrid else "  Speedup (RAG/Hybrid): N/A")
print("=" * 62)

queries_idx = list(range(1, len(rag_times) + 1))

# ── Response Time per Query (per framework) ─────────────────────────────
fig_rt, ax_rt = plt.subplots(figsize=(11, 5))
ax_rt.plot(queries_idx, rag_times, 'o-', color='#3498db', label='RAG',
           linewidth=1.5, markersize=4)
if cag_times:
    ax_rt.plot(queries_idx[:len(cag_times)], cag_times, 's--', color='#2ecc71',
               label='CAG (FAQ only)', linewidth=1.5, markersize=4)
if hybrid_times:
    ax_rt.plot(queries_idx[:len(hybrid_times)], hybrid_times, 'd--', color='#e74c3c',
               label='Hybrid CAG-RAG', linewidth=1.5, markersize=4)
ax_rt.set_title('Response Time per Query', fontsize=14, fontweight='bold')
ax_rt.set_xlabel('Query #', fontsize=12)
ax_rt.set_ylabel('Time (s)', fontsize=12)
ax_rt.tick_params(labelsize=11)
ax_rt.legend(fontsize=11)
ax_rt.grid(axis='y', alpha=0.3)
fig_rt.tight_layout()
fig_rt.savefig('../logs/response_time_perquery_frameworks.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/response_time_perquery_frameworks.png")

# ── Avg Response Time per Framework ────────────────────────────────────
labels, means, stds, colors = [], [], [], []
if pure_cag_m:
    labels.append('CAG')
    means.append(float(pure_cag_m['time_avg']))
    stds.append(float(pure_cag_m['time_std']))
    colors.append('#2ecc71')
labels.append('RAG')
means.append(float(rag_m['time_avg']))
stds.append(float(rag_m['time_std']))
colors.append('#3498db')
if cag_m:
    labels.append('Hybrid')
    means.append(float(cag_m['time_avg']))
    stds.append(float(cag_m['time_std']))
    colors.append('#e74c3c')

fig_avg, ax_avg = plt.subplots(figsize=(7, 5))
bars = ax_avg.bar(labels, means, color=colors,
                  yerr=[s if s > 0 else None for s in stds],
                  capsize=5, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, means):
    ax_avg.text(bar.get_x() + bar.get_width()/2, v + 0.02,
                f'{v:.3f}s', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax_avg.set_title('Avg Response Time per Framework\n(mean ± std)', fontsize=14, fontweight='bold')
ax_avg.set_ylabel('Time (s)', fontsize=12)
ax_avg.tick_params(labelsize=11)
ax_avg.grid(axis='y', alpha=0.3)
fig_avg.tight_layout()
fig_avg.savefig('../logs/avg_response_time_frameworks.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/avg_response_time_frameworks.png")


## Retrieval Inaccuracy Metrics (CAG vs RAG vs Hybrid)

Sesi ini menampilkan metrik Retrieval Inaccuracy sesuai daftar evaluasi utama. Semua metrik berikut tetap dihitung pada pipeline evaluasi:
- Relevance Score
- Exact Match (EM)
- Effective Information Rate (EIR)
- RAG Recall
- BERTScore F1
- Completeness
- Hallucination Rate


In [ ]:
# RETRIEVAL INACCURACY VISUALIZATION — Framework Comparison
# Visual dibagi menjadi 2 bagian, tetapi tetap berada dalam section Retrieval Inaccuracy.

framework_metrics = {
    'CAG': pure_cag_m,
    'RAG': rag_m,
    'HYBRID': cag_m,
}
frameworks = [f for f in ['CAG', 'RAG', 'HYBRID'] if framework_metrics.get(f)]
colors = {'CAG': '#2ecc71', 'RAG': '#3498db', 'HYBRID': '#e74c3c'}
width = 0.75 / max(len(frameworks), 1)


def _safe(v):
    return 0.0 if v is None or (isinstance(v, float) and np.isnan(v)) else v


def _label(v):
    return "N/A" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"


def _plot_grouped(ax, metric_keys, metric_labels, title, ylabel):
    x = np.arange(len(metric_keys))
    for idx, fw in enumerate(frameworks):
        raw_vals = [framework_metrics[fw].get(k) for k in metric_keys]
        vals = [_safe(v) for v in raw_vals]
        pos = x + (idx - (len(frameworks) - 1) / 2) * width
        bars = ax.bar(pos, vals, width, label=fw, color=colors[fw], edgecolor='white')
        for bar, v_raw in zip(bars, raw_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    _label(v_raw), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=11)
    ax.tick_params(labelsize=11)
    ax.grid(axis='y', alpha=0.3)


# ── Part 1: retrieval-oriented metrics ───────────────────────────────────
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 5))
_plot_grouped(
    axes1[0],
    metric_keys=['rel'],
    metric_labels=['Relevance\nScore'],
    title='Relevance Score (higher = better)',
    ylabel='Score',
)
_plot_grouped(
    axes1[1],
    metric_keys=['hall'],
    metric_labels=['Hallucination\nRate'],
    title='Hallucination Rate (lower = better)',
    ylabel='Rate',
)
fig1.suptitle('Retrieval Inaccuracy Metrics: CAG vs RAG vs Hybrid',
              fontsize=13, fontweight='bold', y=1.03)
fig1.tight_layout()
fig1.savefig('../logs/retrieval_inaccuracy_metrics_part1.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/retrieval_inaccuracy_metrics_part1.png")

# ── Part 2: retrieval support metrics ────────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
_plot_grouped(
    axes2[0],
    metric_keys=['recall'],
    metric_labels=['RAG Recall'],
    title='RAG Recall (higher = better)',
    ylabel='Score',
)
_plot_grouped(
    axes2[1],
    metric_keys=['eir'],
    metric_labels=['EIR'],
    title='Effective Information Rate (higher = better)',
    ylabel='Score',
)
fig2.suptitle('Retrieval Inaccuracy Metrics: CAG vs RAG vs Hybrid',
              fontsize=13, fontweight='bold', y=1.03)
fig2.tight_layout()
fig2.savefig('../logs/retrieval_inaccuracy_metrics_part2.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/retrieval_inaccuracy_metrics_part2.png")

In [ ]:
# RETRIEVAL INACCURACY VISUALIZATION — EM / BERTScore / Completeness
# Tetap berada dalam section Retrieval Inaccuracy, bukan section baru.

quality_frameworks = [f for f in ['CAG', 'RAG', 'HYBRID'] if framework_metrics.get(f)]
quality_width = 0.75 / max(len(quality_frameworks), 1)
quality_keys = ['em', 'bert', 'comp']
quality_labels = ['Exact Match (EM)', 'BERTScore F1', 'Completeness']

fig_q, ax_q = plt.subplots(figsize=(8, 5))
x = np.arange(len(quality_keys))
for idx, fw in enumerate(quality_frameworks):
    raw_vals = [framework_metrics[fw].get(k) for k in quality_keys]
    vals = [_safe(v) for v in raw_vals]
    pos = x + (idx - (len(quality_frameworks) - 1) / 2) * quality_width
    bars = ax_q.bar(pos, vals, quality_width, label=fw, color=colors[fw], edgecolor='white')
    for bar, v_raw in zip(bars, raw_vals):
        ax_q.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                  _label(v_raw), ha='center', va='bottom', fontsize=9)

ax_q.set_xticks(x)
ax_q.set_xticklabels(quality_labels, fontsize=11)
ax_q.set_ylabel('Score', fontsize=12)
ax_q.set_title('Retrieval Inaccuracy Metrics (higher = better)', fontsize=13, fontweight='bold')
ax_q.set_ylim(0, 1.1)
ax_q.legend(fontsize=11)
ax_q.tick_params(labelsize=11)
ax_q.grid(axis='y', alpha=0.3)
fig_q.tight_layout()
fig_q.savefig('../logs/retrieval_inaccuracy_metrics_part3.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: logs/retrieval_inaccuracy_metrics_part3.png")

## Statistical Significance Testing
Paired t-test (p < 0.05) + Cohen's d effect size.

In [ ]:
# 🔬 STATISTICAL SIGNIFICANCE TESTING  (Paired t-test + Cohen's d)
if not cag_m:
    print("⚠️  CAG results not available, skipping statistical tests.")
else:
    print("=" * 72)
    print("🔬 STATISTICAL SIGNIFICANCE TESTS (Paired t-test, NaN-aware)")
    print("   H0: No difference between RAG and Hybrid CAG-RAG")
    print("   H1: Significant difference exists (p < 0.05)")
    print("   NaN positions (FAQ path) are excluded from all tests.")
    print("=" * 72)

    # Metrik yang diuji — arah = 'lower' → CAG harus lebih rendah, 'higher' → CAG harus lebih tinggi
    # EM/BERTScore/Completeness tetap dianggap sebagai bagian dari retrieval inaccuracy / answer alignment.
    test_metrics = [
        # (label,              rag_list,             cag_list,             better_direction)
        # ── Latency ──────────────────────────────────────────────────────────
        ("Response Time",      rag_m['_time'],       cag_m['_time'],       "lower"),
        # ── Relevance ─────────────────────────────────────────────────────────
        ("Relevance Score",    rag_m['_rel'],        cag_m['_rel'],        "higher"),
        # ── Retrieval / Answer Alignment ─────────────────────────────────────
        ("Exact Match",        rag_m['_em'],         cag_m['_em'],         "higher"),
        ("EIR",                rag_m['_eir'],        cag_m['_eir'],        "higher"),
        ("RAG Recall",         rag_m['_recall'],     cag_m['_recall'],     "higher"),
        ("BERTScore F1",       rag_m['_bert'],       cag_m['_bert'],       "higher"),
        ("Completeness",       rag_m['_comp'],       cag_m['_comp'],       "higher"),
        # ── Inaccuracy ────────────────────────────────────────────────────────
        ("Hallucination Rate", rag_m['_hall'],       cag_m['_hall'],       "lower"),
    ]

    stat_rows = []
    print(f"\n  {'Metric':<22} {'N':>5} {'RAG µ':>7} {'CAG µ':>7} {'t-stat':>8} {'p-value':>9} {'d':>7}  Result")
    print("  " + "-" * 82)

    for lbl, rag_vals, cag_vals, direction in test_metrics:
        if len(rag_vals) < 2 or len(cag_vals) < 2:
            print(f"  {lbl:<22}  (insufficient data — need ≥ 2 queries)")
            continue

        t_stat, p_val, n_valid = paired_ttest(rag_vals, cag_vals)
        d                      = cohens_d(rag_vals, cag_vals)

        if n_valid < 2:
            print(f"  {lbl:<22}  (no valid pairs after NaN removal — skipped)")
            continue

        # Means computed only on non-NaN values for fair representation
        rag_clean = [x for x in rag_vals if not (isinstance(x, float) and np.isnan(x))]
        cag_clean = [x for x in cag_vals if not (isinstance(x, float) and np.isnan(x))]
        rag_mu    = round(float(np.mean(rag_clean)), 4) if rag_clean else float('nan')
        cag_mu    = round(float(np.mean(cag_clean)), 4) if cag_clean else float('nan')

        significant = not np.isnan(p_val) and p_val < 0.05

        # Check if direction is as expected
        if direction == "lower":
            direction_ok = cag_mu <= rag_mu
        else:
            direction_ok = cag_mu >= rag_mu

        if significant and direction_ok:
            result = "✅ H1 supported"
        elif significant and not direction_ok:
            result = "⚠️  sig. but reversed"
        else:
            result = "❌ not significant"

        n_str = f"{n_valid}" if n_valid == len(rag_vals) else f"{n_valid}*"
        print(f"  {lbl:<22} {n_str:>5} {rag_mu:>7.4f} {cag_mu:>7.4f} "
              f"{t_stat:>8.3f} {p_val:>9.4f} {d:>7.3f}  {result}")
        stat_rows.append({
            'Metric': lbl, 'N': n_valid, 'RAG_mean': rag_mu, 'CAG_mean': cag_mu,
            't_stat': t_stat, 'p_value': p_val, "Cohen_d": d,
            'significant': significant, 'direction_ok': direction_ok
        })

    print("\n  * N < total = beberapa posisi NaN (FAQ path) dikecualikan")
    print("  Legend: ✅ Hypothesis supported  |  ❌ Not significant  |  ⚠️  Unexpected direction")
    print(f"  α = 0.05  |  Effect size: small < 0.2,  medium 0.2–0.8,  large > 0.8")

    # Radar chart for visual comparison
    stat_df = pd.DataFrame(stat_rows)

    # Radar chart
    radar_metrics = ['BERTScore F1', 'Completeness', 'RAG Recall', 'EIR']
    rag_radar  = [rag_m['bert'], rag_m['comp'], rag_m['recall'], rag_m['eir']]
    cag_radar  = [cag_m['bert'], cag_m['comp'], cag_m['recall'], cag_m['eir']]

    # Invert inaccuracy metrics so higher = better on radar
    # Guard against NaN for hallucination (FAQ path excluded from mean already)
    rag_hall_val = rag_m['hall'] if not (isinstance(rag_m['hall'], float) and np.isnan(rag_m['hall'])) else 0.5
    cag_hall_val = cag_m['hall'] if not (isinstance(cag_m['hall'], float) and np.isnan(cag_m['hall'])) else 0.5

    radar_metrics += ['1 - Hallucination']
    rag_radar  += [1 - rag_hall_val]
    cag_radar  += [1 - cag_hall_val]

    # Replace any remaining NaN with 0 for radar chart rendering
    rag_radar = [v if not (isinstance(v, float) and np.isnan(v)) else 0.0 for v in rag_radar]
    cag_radar = [v if not (isinstance(v, float) and np.isnan(v)) else 0.0 for v in cag_radar]

    N   = len(radar_metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    rag_r  = rag_radar + [rag_radar[0]]
    cag_r  = cag_radar + [cag_radar[0]]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, rag_r, 'o-', color='#3498db', linewidth=2, label='RAG')
    ax.fill(angles, rag_r, alpha=0.15, color='#3498db')
    ax.plot(angles, cag_r, 's--', color='#e74c3c', linewidth=2, label='Hybrid CAG-RAG')
    ax.fill(angles, cag_r, alpha=0.15, color='#e74c3c')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_metrics, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title('Radar: Overall Retrieval Inaccuracy Comparison\n(all axes: higher = better)',
                 fontsize=11, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    plt.tight_layout()
    plt.savefig('../logs/radar_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Saved: logs/radar_comparison.png")

## Save & Export

In [ ]:
# Save & Export All Results

import json
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('../logs', exist_ok=True)

speedup_hybrid = (rag_m['time_avg'] / cag_m['time_avg']) if (cag_m and cag_m['time_avg'] > 0) else None
speedup_cag    = (rag_m['time_avg'] / pure_cag_m['time_avg']) if (pure_cag_m and pure_cag_m['time_avg'] > 0) else None

cag_hits = sum(1 for x in pure_cag_res if x.get('cache')) if pure_cag_res else 0
hybrid_hits = sum(1 for x in cag_res if x.get('cache')) if cag_res else 0

# 1. Summary metrics JSON
summary_export = {
    'timestamp':    ts,
    'dataset_size': len(dataset),
    'rag': {k: v for k, v in rag_m.items() if not k.startswith('_') and k != 'per_query'},
    'cag_pure': ({k: v for k, v in pure_cag_m.items() if not k.startswith('_') and k != 'per_query'}
                 if pure_cag_m else None),
    'hybrid': ({k: v for k, v in cag_m.items() if not k.startswith('_') and k != 'per_query'}
               if cag_m else None),
    'speedup_hybrid': round(speedup_hybrid, 4) if speedup_hybrid else None,
    'speedup_cag': round(speedup_cag, 4) if speedup_cag else None,
    'cache_hits': {
        'cag_pure': cag_hits,
        'hybrid': hybrid_hits,
    },
}

json_path = f'../logs/eval_summary_{ts}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_export, f, indent=2, ensure_ascii=False)
print(f"Summary JSON     : {json_path}")

# 2. Per-query CSV — RAG
if rag_m['per_query']:
    rag_csv = f'../logs/eval_rag_perquery_{ts}.csv'
    pd.DataFrame(rag_m['per_query']).to_csv(rag_csv, index=False)
    print(f"RAG per-query    : {rag_csv}")

# 3. Per-query CSV — CAG (pure)
if pure_cag_m and pure_cag_m['per_query']:
    cag_csv = f'../logs/eval_cag_pure_perquery_{ts}.csv'
    pd.DataFrame(pure_cag_m['per_query']).to_csv(cag_csv, index=False)
    print(f"CAG per-query    : {cag_csv}")

# 4. Per-query CSV — Hybrid
if cag_m and cag_m['per_query']:
    hybrid_csv = f'../logs/eval_hybrid_perquery_{ts}.csv'
    pd.DataFrame(cag_m['per_query']).to_csv(hybrid_csv, index=False)
    print(f"Hybrid per-query : {hybrid_csv}")

# 5. Statistical results CSV
if cag_m and 'stat_df' in dir():
    stat_csv = f'../logs/eval_stats_{ts}.csv'
    stat_df.to_csv(stat_csv, index=False)
    print(f"Statistics       : {stat_csv}")

# Key Findings
print("\n" + "=" * 65)
print("KEY FINDINGS  —  CAG vs RAG vs Hybrid CAG-RAG")
print("=" * 65)

print(f"\n  RQ1 LATENCY")
if pure_cag_m:
    print(f"    CAG avg time   : {pure_cag_m['time_avg']:.3f}s +/- {pure_cag_m['time_std']:.3f}s")
    print(f"    CAG CHR        : {pure_cag_m['chr']:.1f}% ({cag_hits}/{len(pure_cag_res)})")
print(f"    RAG avg time   : {rag_m['time_avg']:.3f}s +/- {rag_m['time_std']:.3f}s")
if cag_m:
    print(f"    Hybrid avg time: {cag_m['time_avg']:.3f}s +/- {cag_m['time_std']:.3f}s")
    print(f"    Speedup (RAG/Hybrid): {speedup_hybrid:.2f}x" if speedup_hybrid else "    Speedup (RAG/Hybrid): N/A")
    print(f"    Hybrid CHR     : {cag_m['chr']:.1f}% ({hybrid_hits}/{len(cag_res)})")

print(f"\n  RQ2 INACCURACY")
if cag_m:
    hall_d = rag_m['hall'] - cag_m['hall']
    rel_d  = cag_m['rel']  - rag_m['rel']
    print(f"    Hallucination  RAG={rag_m['hall']:.4f}  Hybrid={cag_m['hall']:.4f}  delta={hall_d:+.4f}")
    print(f"    Relevance      RAG={rag_m['rel']:.4f}  Hybrid={cag_m['rel']:.4f}  delta={rel_d:+.4f}")

    print(f"\n  GROUND-TRUTH ALIGNMENT")
    print(f"    BERTScore F1   RAG={rag_m['bert']:.4f}  Hybrid={cag_m['bert']:.4f}  delta={cag_m['bert']-rag_m['bert']:+.4f}")
    print(f"    Completeness   RAG={rag_m['comp']:.4f}  Hybrid={cag_m['comp']:.4f}  delta={cag_m['comp']-rag_m['comp']:+.4f}")
    print(f"    Exact Match    RAG={rag_m['em']:.4f}  Hybrid={cag_m['em']:.4f}  delta={cag_m['em']-rag_m['em']:+.4f}")

print("\nEvaluation complete.")
print(f"Outputs saved to: logs/")